# Servidor del peaje en RunPod (inferencia M1+M2+M3+pasajeros + agente Qwen)

Ejecuta este notebook en **Jupyter Lab del Pod** de RunPod (no en Colab), celda por celda.

**Antes:** con el botón **Upload** (flecha hacia arriba del panel de archivos de Jupyter) sube a `/workspace` estos archivos de tu PC:
- De `C:\laragon\www\Nueva carpeta\inferencia\`: `servidor_unificado.py`, `requirements.txt`, `runpod_start.sh`, `diagnostico.py`
- `yolo-v9-t-384-license-plates-end2end.onnx` (de `placas-web\python\placas\modelo\`)
- `m2_best.pt` y `llantas_best.pt` (descárgalos de tu Drive: `MyDrive/peaje_modelos/`)

La celda 2 los ordena en `/workspace/inferencia` y `/workspace/modelos`.


In [ ]:
# 1. GPU del Pod
!nvidia-smi


In [ ]:
# 2. Ordenar los archivos subidos a /workspace
import shutil
from pathlib import Path
W = Path('/workspace')
(W / 'inferencia').mkdir(exist_ok=True)
(W / 'modelos').mkdir(exist_ok=True)
for n in ['servidor_unificado.py', 'requirements.txt', 'runpod_start.sh', 'diagnostico.py']:
    if (W / n).exists():
        shutil.move(str(W / n), str(W / 'inferencia' / n))
for n in ['yolo-v9-t-384-license-plates-end2end.onnx', 'm2_best.pt', 'llantas_best.pt']:
    if (W / n).exists():
        shutil.move(str(W / n), str(W / 'modelos' / n))
faltan = [str(p) for p in [W/'inferencia/servidor_unificado.py', W/'inferencia/requirements.txt', W/'inferencia/runpod_start.sh',
          W/'inferencia/diagnostico.py', W/'modelos/yolo-v9-t-384-license-plates-end2end.onnx', W/'modelos/m2_best.pt',
          W/'modelos/llantas_best.pt'] if not p.exists()]
print('Faltan:', faltan if faltan else 'nada ✓')


In [ ]:
# 3. Token (oculto) y arranque en segundo plano: dependencias, diagnóstico, Ollama + Qwen y servidor (10–20 min la primera vez)
import getpass, os, subprocess
token = getpass.getpass('INFERENCE_API_TOKEN (no se mostrará): ').strip()
env = dict(os.environ, INFERENCE_API_TOKEN=token)
log = open('/workspace/servidor.log', 'w')
proc = subprocess.Popen(['bash', 'runpod_start.sh'], cwd='/workspace/inferencia', env=env, stdout=log, stderr=subprocess.STDOUT)
print('Arrancando (PID', proc.pid, '). Revisa el avance con la celda 4.')


In [ ]:
# 4. Avance (vuelve a ejecutar esta celda hasta ver "Uvicorn running on http://0.0.0.0:8000")
print(open('/workspace/servidor.log', errors='replace').read()[-4000:])


In [ ]:
# 5. Comprobar el servidor y mostrar la URL pública para el VPS
import json, os, urllib.request
with urllib.request.urlopen('http://127.0.0.1:8000/', timeout=10) as r:
    print(json.dumps(json.loads(r.read()), indent=2, ensure_ascii=False))
req = urllib.request.Request('http://127.0.0.1:8000/llm/estado', headers={'X-API-Token': token})
with urllib.request.urlopen(req, timeout=10) as r:
    print(json.dumps(json.loads(r.read()), indent=2, ensure_ascii=False))
print('INFERENCE_URL=https://' + os.environ.get('RUNPOD_POD_ID', '<POD_ID>') + '-8000.proxy.runpod.net')


Copia la línea **INFERENCE_URL** y pásala para configurarla en el VPS (`INFERENCE_URL` y `AGENTE_LLM=true`). Deja el Pod encendido durante la demo y el día de la revisión. Log del servidor: `/workspace/servidor.log`.